### Task 1 - Exploratory Data Analysis for AB_NYC_2019.csv Generates diagnostic plots for missing values, price distribution/outliers, minimum_nights outliers, categorical relationships, and geographic patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plt.style.use("seaborn-v0_8-whitegrid")
import os
OUT = "output"
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv("AB_NYC_2019.csv")

miss = df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(miss.index, miss.values, color="#3d6bd4")
for b, v in zip(bars, miss.values):
    ax.text(v + 300, b.get_y() + b.get_height() / 2, f"{v} ({v/len(df)*100:.1f}%)",
            va="center", fontsize=9)
ax.set_xlabel("Missing count")
ax.set_title("Missing values by column")
ax.set_xlim(0, miss.max() * 1.25)
plt.tight_layout()
plt.savefig(f"{OUT}/01_missing_values.png", dpi=140)
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["price"], bins=200, color="#3d6bd4")
axes[0].set_xlim(0, 1000)
axes[0].set_title("Raw price distribution (0-1000)")
axes[0].set_xlabel("price ($)")
axes[0].set_ylabel("count")

log_price = np.log1p(df["price"])
axes[1].hist(log_price, bins=80, color="#d4763d")
axes[1].set_title("log1p(price) distribution")
axes[1].set_xlabel("log(1 + price)")

axes[2].boxplot(df["price"], vert=True, showfliers=True,
                 flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[2].set_yscale("log")
axes[2].set_title("Price boxplot (log scale)\nshowing outliers")
axes[2].set_ylabel("price ($, log scale)")

plt.tight_layout()
plt.savefig(f"{OUT}/02_price_distribution.png", dpi=140)
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

order1 = df.groupby("room_type")["price"].median().sort_values(ascending=False).index
data1 = [df.loc[df.room_type == rt, "price"].clip(upper=500) for rt in order1]
axes[0].boxplot(data1, tick_labels=order1, showfliers=False)
axes[0].set_title("Price by room type (capped at $500, outliers hidden)")
axes[0].set_ylabel("price ($)")
axes[0].tick_params(axis='x', rotation=15)

order2 = df.groupby("neighbourhood_group")["price"].median().sort_values(ascending=False).index
data2 = [df.loc[df.neighbourhood_group == ng, "price"].clip(upper=500) for ng in order2]
axes[1].boxplot(data2, tick_labels=order2, showfliers=False)
axes[1].set_title("Price by borough (capped at $500, outliers hidden)")
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(f"{OUT}/03_price_by_category.png", dpi=140)
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["minimum_nights"].clip(upper=60), bins=60, color="#3d6bd4")
axes[0].set_title("minimum_nights (clipped at 60)")
axes[0].set_xlabel("minimum_nights")

axes[1].boxplot(df["minimum_nights"], showfliers=True,
                 flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].set_yscale("log")
axes[1].set_title(f"minimum_nights boxplot (log scale)\nmax={df.minimum_nights.max()}, "
                   f">365 nights: {(df.minimum_nights>365).sum()} rows")
plt.tight_layout()
plt.savefig(f"{OUT}/04_minimum_nights_outliers.png", dpi=140)
plt.close()

from PIL import Image
img = np.array(Image.open("New_York_City_.png").convert("RGB"))
# Approximate geographic bounding box of the supplied borough map image
# (covers all 5 boroughs incl. Staten Island / Rockaways)
extent = [-74.28, -73.68, 40.47, 40.93]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

for ax, (col, cmap, title, vmax) in zip(
    axes,
    [("price", "viridis", "Price by location (capped at $400)", 400),
     ("room_type", None, "Room type by location", None)]
):
    ax.imshow(img, extent=extent, aspect="auto")
    if col == "price":
        sc = ax.scatter(df["longitude"], df["latitude"], c=df["price"].clip(upper=vmax),
                         cmap=cmap, s=3, alpha=0.5, vmin=0, vmax=vmax)
        plt.colorbar(sc, ax=ax, label="price ($, capped)", fraction=0.04)
    else:
        for rt, color in zip(df.room_type.unique(), ["#e74c3c", "#3498db", "#2ecc71"]):
            sub = df[df.room_type == rt]
            ax.scatter(sub["longitude"], sub["latitude"], s=3, alpha=0.4, label=rt, color=color)
        ax.legend(markerscale=4, loc="upper left")
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_title(title)
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")

plt.tight_layout()
plt.savefig(f"{OUT}/05_geo_scatter.png", dpi=140)
plt.close()


num_cols = ["price", "minimum_nights", "number_of_reviews", "reviews_per_month",
            "calculated_host_listings_count", "availability_365", "latitude", "longitude"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha="right")
ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Correlation matrix (numeric features)")
plt.tight_layout()
plt.savefig(f"{OUT}/06_correlation_heatmap.png", dpi=140)
plt.close()

print("All plots saved to", OUT)

C:\Users\Admin\AppData\Local\Temp\ipykernel_21416\391126438.py:53: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  axes[2].boxplot(df["price"], vert=True, showfliers=True,


All plots saved to output


In [ ]:
# Drop columns that are either identifiers, text-heavy, or redundant for baseline modeling
cols_to_drop = ['id', 'name', 'host_name', 'last_review']
df_clean = df.drop(columns=cols_to_drop)

# Fill missing values in 'reviews_per_month'. 
# NaN here means 0 reviews, so we impute with 0.
df_clean['reviews_per_month'] = df_clean['reviews_per_month'].fillna(0)

print("Missing values after cleaning:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# Drop listings with a price of $0 (likely data entry errors)
df_clean = df_clean[df_clean['price'] > 0]

# Cap extreme upper-bound outliers for 'price' at the 99th percentile (~$799)
price_99th = df_clean['price'].quantile(0.99)
df_clean['price'] = np.where(df_clean['price'] > price_99th, price_99th, df_clean['price'])

# Cap 'minimum_nights' at 365 (1 year)
df_clean['minimum_nights'] = np.where(df_clean['minimum_nights'] > 365, 365, df_clean['minimum_nights'])

# The EDA showed 'price' is highly right-skewed. 
# We apply a log1p transformation to make it more normally distributed for linear models.
df_clean['log_price'] = np.log1p(df_clean['price'])

# Frequency encoding for the high-cardinality 'neighbourhood' column
neighbourhood_freq = df_clean['neighbourhood'].value_counts() / len(df_clean)
df_clean['neighbourhood_freq'] = df_clean['neighbourhood'].map(neighbourhood_freq)

# Drop the original 'neighbourhood' text column now that it's encoded
df_clean = df_clean.drop(columns=['neighbourhood'])

# One-hot encode low-cardinality categorical variables
categorical_features = ['neighbourhood_group', 'room_type']
df_final = pd.get_dummies(df_clean, columns=categorical_features, drop_first=True)

print("\nFinal dataset shape for modeling:", df_final.shape)
print(df_final.head())

Missing values after cleaning:
Series([], dtype: int64)

Final dataset shape for modeling: (48884, 17)
   host_id  latitude  longitude  price  minimum_nights  number_of_reviews  \
0     2787  40.64749  -73.97237  149.0               1                  9   
1     2845  40.75362  -73.98377  225.0               1                 45   
2     4632  40.80902  -73.94190  150.0               3                  0   
3     4869  40.68514  -73.95976   89.0               1                270   
4     7192  40.79851  -73.94399   80.0              10                  9   

   reviews_per_month  calculated_host_listings_count  availability_365  \
0               0.21                               6               365   
1               0.38                               2               355   
2               0.00                               1               365   
3               4.64                               1               194   
4               0.10                               1            

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot capped price
axes[0].hist(df_clean['price'], bins=50, color='#2ecc71')
axes[0].set_title('Cleaned Price Distribution (Capped at 99th Percentile)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

# Plot log transformed price
axes[1].hist(df_clean['log_price'], bins=50, color='#9b59b6')
axes[1].set_title('Log-Transformed Price Distribution')
axes[1].set_xlabel('log(1 + Price)')

plt.tight_layout()
plt.savefig(f"{OUT}/07_cleaned_distributions.png", dpi=140)
plt.show()

from sklearn.model_selection import train_test_split

# Drop 'host_id' as it is an identifier and won't help the model generalize
if 'host_id' in df_final.columns:
    df_final = df_final.drop(columns=['host_id'])

# Separate features (X) and target (y)
# We predict 'log_price' to improve linear/distance-based model performance
X = df_final.drop(columns=['price', 'log_price'])
y = df_final['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training features shape:", X_train.shape)

C:\Users\Admin\AppData\Local\Temp\ipykernel_21416\3963624880.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Training features shape: (39107, 14)


### Task 2 - Model Training and Evaluation

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(name, y_true_log, y_pred_log):
    # Convert predictions back to dollars using expm1 for interpretable metrics
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"{name} - RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R2: {r2:.4f}")
    return rmse, mae, r2

# 1. Baseline: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
print("\n--- Baseline ---")
_ = evaluate_model("Linear Regression", y_test, lr.predict(X_test))

# 2. Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("\n--- Ensemble ---")
_ = evaluate_model("Random Forest", y_test, rf.predict(X_test))

# 3. XGBoost Regressor
xgb = XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
print("\n--- Gradient Boosting ---")
_ = evaluate_model("XGBoost", y_test, xgb.predict(X_test))


--- Baseline ---
Linear Regression - RMSE: $103.99 | MAE: $54.56 | R2: 0.2710

--- Ensemble ---
Random Forest - RMSE: $93.25 | MAE: $48.34 | R2: 0.4138

--- Gradient Boosting ---
XGBoost - RMSE: $93.14 | MAE: $48.27 | R2: 0.4151

Starting Hyperparameter Tuning for XGBoost...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best XGBoost Parameters: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 1.0}

--- Tuned Model Performance ---
Tuned XGBoost - RMSE: $93.08 | MAE: $47.79 | R2: 0.4159

Saved 'airbnb_xgb_model.pkl' and 'model_features.pkl' successfully.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import joblib

# Metric helper to assess performance and check for overfitting/underfitting
def get_metrics(model, X_tr, y_tr, X_te, y_te):
    # Train performance
    y_tr_pred = np.expm1(model.predict(X_tr))
    y_tr_true = np.expm1(y_tr)
    train_rmse = np.sqrt(mean_squared_error(y_tr_true, y_tr_pred))
    train_r2 = r2_score(y_tr_true, y_tr_pred)
    
    # Test performance
    y_te_pred = np.expm1(model.predict(X_te))
    y_te_true = np.expm1(y_te)
    test_rmse = np.sqrt(mean_squared_error(y_te_true, y_te_pred))
    test_mae = mean_absolute_error(y_te_true, y_te_pred)
    test_r2 = r2_score(y_te_true, y_te_pred)
    
    return {
        "Train RMSE": train_rmse,
        "Train R2": train_r2,
        "Test RMSE": test_rmse,
        "Test MAE": test_mae,
        "Test R2": test_r2
    }

# --- A. Random Forest Tuning ---
print("Tuning Random Forest Regressor...")
rf_param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.8]
}

rf_cv = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=rf_param_grid,
    n_iter=8,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rf_cv.fit(X_train, y_train)
best_rf = rf_cv.best_estimator_
print(f"Best RF Parameters: {rf_cv.best_params_}")

# --- B. XGBoost Tuning ---
print("\nTuning XGBoost Regressor...")
xgb_param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_cv = RandomizedSearchCV(
    estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
    param_distributions=xgb_param_grid,
    n_iter=8,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

xgb_cv.fit(X_train, y_train)
best_xgb = xgb_cv.best_estimator_
print(f"Best XGBoost Parameters: {xgb_cv.best_params_}")


rf_results = get_metrics(best_rf, X_train, y_train, X_test, y_test)
xgb_results = get_metrics(best_xgb, X_train, y_train, X_test, y_test)

comparison_df = pd.DataFrame([rf_results, xgb_results], index=["Tuned Random Forest", "Tuned XGBoost"])
print("\nModel Comparison Table:")
print(comparison_df.round(3))

Tuning Random Forest Regressor...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best RF Parameters: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}

Tuning XGBoost Regressor...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best XGBoost Parameters: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

Model Comparison Table:
                     Train RMSE  Train R2  Test RMSE  Test MAE  Test R2
Tuned Random Forest      68.237     0.687     93.963    47.653    0.405
Tuned XGBoost            86.416     0.498     94.390    48.070    0.399


In [ ]:
# Automatically save the model with the higher test R2
if xgb_results["Test R2"] >= rf_results["Test R2"]:
    best_model = best_xgb
    best_name = "Tuned XGBoost"
else:
    best_model = best_rf
    best_name = "Tuned Random Forest"

joblib.dump(best_model, 'best_airbnb_model.pkl')
joblib.dump(list(X.columns), 'model_features.pkl')

print(f"\n{best_name} selected as best model.")
print("Saved 'best_airbnb_model.pkl' and 'model_features.pkl' for the app.")


Tuned Random Forest selected as best model.
Saved 'best_airbnb_model.pkl' and 'model_features.pkl' for the app.
